# 03_observability_queries

Crea las vistas SQL necesarias para el dashboard de observabilidad FinPay.

Corrección: `details` ya viene como JSON en STRING, por eso se usa `get_json_object(details, ...)`.


Parámetros:
- `catalog`: catálogo objetivo. Usar `fintech_finpay_dev` para DEV y `fintech_finpay` para PROD.


In [ ]:
USE CATALOG fintech_finpay;


In [ ]:
-- 03_observability_queries.ipynb
-- Crea las vistas que alimentan el dashboard de observabilidad FinPay.
-- Corrección: en tu event log, details ya es STRING/JSON, por eso NO usamos to_json(details).

CREATE SCHEMA IF NOT EXISTS observability;


In [ ]:
CREATE OR REPLACE VIEW observability.v_pipeline_events AS
SELECT
  timestamp AS event_timestamp,
  DATE(timestamp) AS event_date,
  event_type,
  level,
  message,
  origin,
  details
FROM observability.finpay_etl_pipeline_event_log;


In [ ]:
CREATE OR REPLACE VIEW observability.v_flow_progress AS
SELECT
  timestamp AS event_timestamp,
  DATE(timestamp) AS event_date,
  event_type,
  get_json_object(details, '$.flow_progress.name') AS flow_name,
  CASE
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'bronze.%' THEN 'Bronze'
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'silver.%' THEN 'Silver'
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'gold.%' THEN 'Gold'
    ELSE 'Other'
  END AS layer,
  get_json_object(details, '$.flow_progress.status') AS flow_status,
  CAST(get_json_object(details, '$.flow_progress.metrics.num_input_rows') AS BIGINT) AS input_rows,
  CAST(get_json_object(details, '$.flow_progress.metrics.num_output_rows') AS BIGINT) AS output_rows,
  get_json_object(details, '$.flow_progress.metrics') AS metrics_json,
  get_json_object(details, '$.flow_progress.data_quality') AS data_quality_json
FROM observability.finpay_etl_pipeline_event_log
WHERE event_type = 'flow_progress';


In [ ]:
CREATE OR REPLACE VIEW fintech_finpay.observability.v_processed_records_by_layer AS
SELECT
  current_date() AS event_date,
  'Bronze' AS layer,
  'bronze.transactions' AS flow_name,
  'COMPLETED' AS flow_status,
  COUNT(*) AS input_records,
  COUNT(*) AS processed_records
FROM fintech_finpay.bronze.transactions

UNION ALL
SELECT
  current_date(),
  'Bronze',
  'bronze.merchants',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.bronze.merchants

UNION ALL
SELECT
  current_date(),
  'Bronze',
  'bronze.users',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.bronze.users

UNION ALL
SELECT
  current_date(),
  'Bronze',
  'bronze.transactions_valid_changes',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.bronze.transactions_valid_changes

UNION ALL
SELECT
  current_date(),
  'Bronze',
  'bronze.merchants_valid_changes',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.bronze.merchants_valid_changes

UNION ALL
SELECT
  current_date(),
  'Bronze',
  'bronze.users_valid_changes',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.bronze.users_valid_changes

UNION ALL
SELECT
  current_date(),
  'Silver',
  'silver.transactions',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.silver.transactions

UNION ALL
SELECT
  current_date(),
  'Silver',
  'silver.merchants',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.silver.merchants

UNION ALL
SELECT
  current_date(),
  'Silver',
  'silver.users',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.silver.users

UNION ALL
SELECT
  current_date(),
  'Gold',
  'gold.transactions_enriched',
  'COMPLETED',
  COUNT(*),
  COUNT(*)
FROM fintech_finpay.gold.transactions_enriched;


In [ ]:
CREATE OR REPLACE VIEW observability.v_quality_expectations AS
SELECT
  timestamp AS event_timestamp,
  DATE(timestamp) AS event_date,
  get_json_object(details, '$.flow_progress.name') AS flow_name,
  CASE
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'bronze.%' THEN 'Bronze'
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'silver.%' THEN 'Silver'
    WHEN get_json_object(details, '$.flow_progress.name') LIKE 'gold.%' THEN 'Gold'
    ELSE 'Other'
  END AS layer,
  get_json_object(details, '$.flow_progress.data_quality.expectations') AS expectations_json
FROM observability.finpay_etl_pipeline_event_log
WHERE event_type = 'flow_progress'
  AND get_json_object(details, '$.flow_progress.data_quality.expectations') IS NOT NULL;


In [ ]:
CREATE OR REPLACE VIEW observability.v_rejected_records AS
SELECT
  load_date AS event_date,
  'Silver' AS layer,
  source_name,
  target_table,
  rejection_reason,
  COUNT(*) AS rejected_records
FROM silver.quarantine
GROUP BY load_date, source_name, target_table, rejection_reason;


In [ ]:
CREATE OR REPLACE VIEW fintech_finpay.observability.v_rejection_rate_by_layer AS
WITH processed AS (
  SELECT
    event_date,
    layer,
    SUM(processed_records) AS processed_records
  FROM fintech_finpay.observability.v_processed_records_by_layer
  GROUP BY event_date, layer
),
rejected AS (
  SELECT
    event_date,
    layer,
    SUM(rejected_records) AS rejected_records
  FROM fintech_finpay.observability.v_rejected_records
  GROUP BY event_date, layer
)
SELECT
  COALESCE(p.event_date, r.event_date) AS event_date,
  COALESCE(p.layer, r.layer) AS layer,
  COALESCE(p.processed_records, 0) AS processed_records,
  COALESCE(r.rejected_records, 0) AS rejected_records,
  CASE
    WHEN COALESCE(p.processed_records, 0) = 0 THEN 0.0
    ELSE CAST(COALESCE(r.rejected_records, 0) AS DOUBLE) / CAST(p.processed_records AS DOUBLE)
  END AS rejection_rate
FROM processed p
FULL OUTER JOIN rejected r
  ON p.event_date = r.event_date
 AND p.layer = r.layer;

In [ ]:
CREATE OR REPLACE VIEW observability.v_observability_summary AS
SELECT
  event_date,
  layer,
  SUM(processed_records) AS processed_records,
  0 AS rejected_records,
  0.0 AS rejection_rate
FROM observability.v_processed_records_by_layer
GROUP BY event_date, layer

UNION ALL

SELECT
  event_date,
  layer,
  0 AS processed_records,
  SUM(rejected_records) AS rejected_records,
  0.0 AS rejection_rate
FROM observability.v_rejected_records
GROUP BY event_date, layer;


In [ ]:
-- Validación rápida de fuentes para el dashboard.

SELECT 'v_processed_records_by_layer' AS object_name, COUNT(*) AS total
FROM observability.v_processed_records_by_layer
UNION ALL
SELECT 'v_rejected_records' AS object_name, COUNT(*) AS total
FROM observability.v_rejected_records
UNION ALL
SELECT 'v_rejection_rate_by_layer' AS object_name, COUNT(*) AS total
FROM observability.v_rejection_rate_by_layer;
